# Stage 3 — Semantic chunking + embeddings

Input: `data/cleaned/{text,table,image}_units.jsonl` from `2_clean_and_organize.ipynb`.

This notebook:

1. Captions every image with a vision LLM (Azure OpenAI GPT-4o) — fills the `content: null` gap left in Stage 2
2. Semantically chunks text **per page** (embedding-distance breakpoints between sentences, not fixed size)
3. Passes tables through unchanged (already one atomic chunk each)
4. Unifies all three into one `chunks` list with a shared schema and a `chunk_id`
5. Embeds every chunk's content with Azure OpenAI's embedding model
6. Saves everything to `data/chunks/chunks.jsonl` — the direct input to Stage 4 (Milvus + the three indexes)

**Needs an Azure OpenAI resource** with a GPT-4o deployment and an embedding-model deployment (e.g. `text-embedding-3-small`), and these `.env` variables:

```text
AZURE_OPENAI_ENDPOINT=...
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_API_VERSION=2024-08-01-preview
AZURE_OPENAI_GPT4O_DEPLOYMENT=<your gpt-4o deployment name>
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=<your embedding deployment name>
```


In [1]:
import base64
import hashlib
import json
import os
import re
import time
import uuid
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import tiktoken
from dotenv import load_dotenv
from openai import AzureOpenAI
from tqdm.auto import tqdm

from langchain_openai import AzureOpenAIEmbeddings

/Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CLEANED_DIR = DATA_DIR / "cleaned"
CAPTION_CACHE_DIR = DATA_DIR / "caption_cache"
CHUNKS_DIR = DATA_DIR / "chunks"
LOGS_DIR = DATA_DIR / "logs"

for d in (CAPTION_CACHE_DIR, CHUNKS_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

MIN_CHARS_FOR_SEMANTIC_CHUNKING = 300  # pages with less text than this skip the chunker and stay one chunk

usage_log: list[dict] = []


def record_usage(call_type: str, n_items: int, latency_sec: float, prompt_tokens: int = 0, completion_tokens: int = 0) -> None:
    """Append one API-call record. Only call this for an actual network call, never a cache hit."""
    usage_log.append({
        "call_type": call_type,
        "n_items": n_items,
        "latency_sec": latency_sec,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": prompt_tokens + completion_tokens,
    })


def load_jsonl(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


text_units = load_jsonl(CLEANED_DIR / "text_units.jsonl")
table_units = load_jsonl(CLEANED_DIR / "table_units.jsonl")
image_units = load_jsonl(CLEANED_DIR / "image_units.jsonl")

print(f"text_units:  {len(text_units)}")
print(f"table_units: {len(table_units)}")
print(f"image_units: {len(image_units)}")

text_units:  9563
table_units: 25
image_units: 71


## Image captioning (Azure OpenAI GPT-4o)

Each cropped figure is sent to GPT-4o as a base64-encoded image with a prompt asking for a retrieval-oriented description (chart type, variables/axes, notable trend or values) — this becomes the image chunk's `content`.

Captions are cached to `data/caption_cache/` keyed by an md5 hash of `image_path`, same rationale as the DI cache in Stage 1: GPT-4o vision calls are billed, so a re-run should never re-caption an image it already captioned.


In [3]:
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_API_VERSION = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
AZURE_OPENAI_GPT4O_DEPLOYMENT = os.environ["AZURE_OPENAI_GPT4O_DEPLOYMENT"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]

CAPTION_PROMPT = (
    "You are describing a figure from an IPCC climate science report chapter for a search index. "
    "Describe what this chart or diagram shows: the type of visualization, the variables/axes involved, "
    "and any notable trend, value, or labeled region. Be specific and factual, 3-5 sentences. "
    "Do not start with phrases like 'This image shows' — just describe the content directly."
)


def get_openai_client() -> AzureOpenAI:
    return AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
    )


def caption_image_cached(image_path: str, client: AzureOpenAI) -> str:
    cache_path = CAPTION_CACHE_DIR / f"{hashlib.md5(image_path.encode()).hexdigest()}.txt"
    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8")

    b64 = base64.b64encode((PROJECT_ROOT / image_path).read_bytes()).decode("utf-8")

    start = time.perf_counter()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_GPT4O_DEPLOYMENT,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": CAPTION_PROMPT},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                ],
            }
        ],
        max_tokens=300,
    )
    latency = time.perf_counter() - start

    record_usage(
        "caption",
        n_items=1,
        latency_sec=latency,
        prompt_tokens=response.usage.prompt_tokens,
        completion_tokens=response.usage.completion_tokens,
    )

    caption = response.choices[0].message.content.strip()
    cache_path.write_text(caption, encoding="utf-8")
    return caption

#### Sanity check on one image before captioning all of them"

In [4]:
_openai_client = get_openai_client()
_test_caption = caption_image_cached(image_units[0]["image_path"], _openai_client)
print(_test_caption)

Chapter 7 of the IPCC climate science report is outlined through a structured table presenting categorized sections and key topics labeled with corresponding sub-section numbers and FAQ references. The left panel lists six main sections discussing concepts such as the energy budget, radiative forcing, climate feedbacks, climate sensitivity, and associated metrics. The right panel provides a quick guide linking specific topics—including aerosols, polar amplification, and paleoclimate data—to their detailed discussion in the chapter or FAQs. At the bottom, additional boxes are indicated for deeper exploration of frameworks, global energy budgets, and cross-chapter topics.


In [5]:
for unit in tqdm(image_units, desc="captioning images"):
    unit["content"] = caption_image_cached(unit["image_path"], _openai_client)

pd.DataFrame(image_units)[["doc_id", "page_number", "content"]].head(5)

captioning images:   0%|          | 0/71 [00:00<?, ?it/s]

captioning images: 100%|██████████| 71/71 [00:00<00:00, 4370.86it/s]


,doc_id,page_number,content
0,IPCC_AR6_WGI_Chapter07.pdf,7,Chapter 7 of the IPCC climate science report i...
1,IPCC_AR6_WGI_Chapter07.pdf,8,A conceptual diagram illustrates components in...
2,IPCC_AR6_WGI_Chapter07.pdf,8,A schematic diagram highlights three categorie...
3,IPCC_AR6_WGI_Chapter07.pdf,10,Panel (a) shows Earth's energy imbalance (W/m²...
4,IPCC_AR6_WGI_Chapter07.pdf,12,Two schematic diagrams illustrate Earth’s ener...


## Embedding model + semantic chunking

Embeddings come from Azure OpenAI (`AZURE_OPENAI_EMBEDDING_DEPLOYMENT`), same resource as the GPT-4o captioning above. LangChain's `AzureOpenAIEmbeddings` doesn't surface token usage the way a raw API response does, so `embed_with_logging()` wraps every call with a timer and a `tiktoken` token count instead — an estimate, not billing-exact, but close enough to track cost and throughput.

`langchain-experimental` (where `SemanticChunker` lived) is being sunset, so semantic chunking is implemented directly here instead — it's a short, well-understood algorithm:

1. Split a page's text into sentences
2. Embed every sentence — `embed_documents` batches a page's sentences into one API call, so this isn't one network round-trip per sentence
3. Compute the cosine distance between each pair of consecutive sentences
4. Mark a breakpoint wherever that distance exceeds the `BREAKPOINT_PERCENTILE`-th percentile of all distances on that page — a statistically unusual jump signals a topic shift
5. Group sentences between breakpoints into one chunk each

Same technique as before — sentence-level embedding similarity with a coherence-drop cutoff — just backed by Azure OpenAI embeddings instead of a local model.


In [6]:
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
)

_token_encoder = tiktoken.encoding_for_model("text-embedding-3-small")  # tokenizer estimate only, not tied to your exact deployment


def embed_with_logging(texts: list[str]) -> list[list[float]]:
    start = time.perf_counter()
    vectors = embeddings.embed_documents(texts)
    latency = time.perf_counter() - start

    token_estimate = sum(len(_token_encoder.encode(t)) for t in texts)
    record_usage("embedding", n_items=len(texts), latency_sec=latency, prompt_tokens=token_estimate)

    return vectors


SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+(?=[A-Z(])")
BREAKPOINT_PERCENTILE = 95  # sentence-pair distances above this percentile mark a new chunk


def split_sentences(text: str) -> list[str]:
    """Regex sentence split — will occasionally over-split on abbreviations like 'Fig. 7.2',
    which just produces a slightly finer chunk boundary there, not an incorrect one."""
    return [s.strip() for s in SENTENCE_SPLIT_RE.split(text.strip()) if s.strip()]


def cosine_distance(a: list[float], b: list[float]) -> float:
    a, b = np.array(a), np.array(b)
    similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    return 1 - float(similarity)


def semantic_chunk_page(page_text: str) -> list[str]:
    sentences = split_sentences(page_text)
    if len(sentences) <= 1:
        return [page_text.strip()] if page_text.strip() else []

    vectors = embed_with_logging(sentences)
    distances = [cosine_distance(vectors[i], vectors[i + 1]) for i in range(len(vectors) - 1)]
    threshold = np.percentile(distances, BREAKPOINT_PERCENTILE)

    chunks, current = [], [sentences[0]]
    for i, dist in enumerate(distances):
        if dist > threshold:
            chunks.append(" ".join(current))
            current = [sentences[i + 1]]
        else:
            current.append(sentences[i + 1])
    chunks.append(" ".join(current))
    return chunks

## Semantic chunking — text, per page

Text units are grouped by `(doc_id, page_number)` and concatenated in their original reading order into one block per page, then chunked. Chunking **per page** (rather than per whole document) means every resulting chunk keeps a single, exact page number — important for citations later — at the small cost of not letting a semantic chunk span a page break.

Pages with very little text (`< MIN_CHARS_FOR_SEMANTIC_CHUNKING`) skip the chunker entirely and become one chunk — too little text for sentence-level breakpoint detection to mean anything.


In [7]:
pages = defaultdict(list)
for u in text_units:
    pages[(u["doc_id"], u["page_number"])].append(u)

print(f"{len(pages)} pages with text across {len(text_units)} paragraphs")

288 pages with text across 9563 paragraphs


In [8]:
text_chunks = []

for (doc_id, page_number), group in tqdm(pages.items(), desc="semantic chunking pages"):
    page_text = "\n\n".join(u["content"] for u in group)

    if len(page_text) < MIN_CHARS_FOR_SEMANTIC_CHUNKING:
        pieces = [page_text]
    else:
        pieces = semantic_chunk_page(page_text)

    for piece in pieces:
        piece = piece.strip()
        if not piece:
            continue
        text_chunks.append({
            "doc_id": doc_id,
            "page_number": page_number,
            "chunk_type": "text",
            "content": piece,
        })

print(f"{len(text_chunks)} text chunks from {len(pages)} pages")

semantic chunking pages: 100%|██████████| 288/288 [02:46<00:00,  1.73it/s]

1205 text chunks from 288 pages


## Table and image chunks

No further splitting needed — each table is already one atomic unit (splitting rows would break its structure), and each image already has exactly one caption from the step above. Both just get reshaped into the same chunk schema as the text chunks.


In [9]:
table_chunks = [
    {
        "doc_id": u["doc_id"],
        "page_number": u["page_number"],
        "chunk_type": "table",
        "content": u["content"],
    }
    for u in table_units
]

image_chunks = [
    {
        "doc_id": u["doc_id"],
        "page_number": u["page_number"],
        "chunk_type": "image",
        "content": u["content"],
        "image_path": u["image_path"],
    }
    for u in image_units
]

print(f"table_chunks: {len(table_chunks)}")
print(f"image_chunks: {len(image_chunks)}")

table_chunks: 25
image_chunks: 71


#### Unify into one chunk list

All three streams share the same core schema now, so they merge into a single list. Each chunk gets a `chunk_id` (uuid4 hex) — this becomes the primary key when inserting into Milvus in Stage 4.


In [10]:
all_chunks = text_chunks + table_chunks + image_chunks

for chunk in all_chunks:
    chunk["chunk_id"] = uuid.uuid4().hex

print(f"total chunks: {len(all_chunks)}")
pd.DataFrame(all_chunks)["chunk_type"].value_counts()

total chunks: 1301


chunk_type
text     1205
image      71
table      25
Name: count, dtype: int64

## Embed every chunk

All chunk content (prose, table Markdown, image captions alike) is embedded with the same Azure OpenAI deployment used above, via `embed_with_logging` so this call is captured in `usage_log` too. `embed_documents` batches the whole call into as few requests as the API allows rather than one call per chunk.

The printed dimension is what fixes the vector field size in Milvus's collection schema in Stage 4.


In [11]:
contents = [c["content"] for c in all_chunks]
vectors = embed_with_logging(contents)

for chunk, vector in zip(all_chunks, vectors):
    chunk["embedding"] = vector

EMBEDDING_DIM = len(vectors[0])
print(f"embedded {len(all_chunks)} chunks, dimension = {EMBEDDING_DIM}")

embedded 1301 chunks, dimension = 1536


In [12]:
def save_jsonl(units: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for unit in units:
            f.write(json.dumps(unit) + "\n")


save_jsonl(all_chunks, CHUNKS_DIR / "chunks.jsonl")
print(f"saved {len(all_chunks)} chunks to {CHUNKS_DIR / 'chunks.jsonl'}")

saved 1301 chunks to /Users/rutikakadam/Documents/My Studies/Projects/multi-modal-rag/data/chunks/chunks.jsonl


In [13]:
preview_df = pd.DataFrame(all_chunks)[["chunk_id", "doc_id", "page_number", "chunk_type", "content"]]
preview_df.groupby("chunk_type").sample(2, random_state=0)

,chunk_id,doc_id,page_number,chunk_type,content
1256,28e3fb3f778342c18948628eaf98bafd,IPCC_AR6_WGI_Chapter07.pdf,84,image,Two panels compare assessments of the equilibr...
1257,9c6b41b102c34f40a3882cdb70d5ddb3,IPCC_AR6_WGI_Chapter07.pdf,87,image,Five panels visualize temperature anomalies (°...
1214,8a7a44f2bd274250968fb23178b1359f,IPCC_AR6_WGI_Chapter07.pdf,38,table,| Driver | Global Mean Effective Radiative For...
1221,b3b3f6a4e95d4a018cd59c34d55f9a13,IPCC_AR6_WGI_Chapter07.pdf,80,table,| (1) Study | (2) Time Period (kyr = thousand ...
661,45ecf2d9748349d8813742d66703fd13,IPCC_AR6_WGI_Chapter08.pdf,52,text,"Over South America, a positive SAM is associat..."
472,305a773b42ed434193e1d34352f4d46e,IPCC_AR6_WGI_Chapter07.pdf,126,text,"Allen, 2021: Further improvement of warmingequ..."


## Usage summary — cost, latency, throughput

Every captioning and embedding call in this notebook was logged to `usage_log` (cache hits excluded, since those made no network call). This aggregates it into per-call-type totals and an estimated cost.

**`PRICE_PER_1K_TOKENS` below are placeholder rates — verify against the current [Azure OpenAI pricing page](https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/) before trusting the dollar figure.** The token/latency/throughput numbers are exact (or a close `tiktoken` estimate for embeddings); only the cost conversion depends on rates that change over time.


In [14]:
PRICE_PER_1K_TOKENS = {
    ("caption", "prompt"): 0.0025,      # gpt-4o input -- VERIFY against current pricing
    ("caption", "completion"): 0.01,    # gpt-4o output -- VERIFY against current pricing
    ("embedding", "prompt"): 0.00002,   # text-embedding-3-small -- VERIFY against current pricing
}

save_jsonl(usage_log, LOGS_DIR / "stage3_usage_log.jsonl")

if not usage_log:
    print("no new API calls were made this run (everything served from cache)")
else:
    usage_df = pd.DataFrame(usage_log)
    usage_df["estimated_cost_usd"] = (
        usage_df["prompt_tokens"] / 1000 * usage_df["call_type"].map(lambda c: PRICE_PER_1K_TOKENS.get((c, "prompt"), 0))
        + usage_df["completion_tokens"] / 1000 * usage_df["call_type"].map(lambda c: PRICE_PER_1K_TOKENS.get((c, "completion"), 0))
    )

    summary = usage_df.groupby("call_type").agg(
        calls=("call_type", "count"),
        total_items=("n_items", "sum"),
        total_tokens=("total_tokens", "sum"),
        total_latency_sec=("latency_sec", "sum"),
        avg_latency_sec=("latency_sec", "mean"),
        estimated_cost_usd=("estimated_cost_usd", "sum"),
    )
    summary["items_per_sec"] = summary["total_items"] / summary["total_latency_sec"]
    summary["tokens_per_sec"] = summary["total_tokens"] / summary["total_latency_sec"]

    display(summary)
    print(f"total estimated cost this run: ${usage_df['estimated_cost_usd'].sum():.4f}")

,calls,total_items,total_tokens,total_latency_sec,avg_latency_sec,estimated_cost_usd,items_per_sec,tokens_per_sec
call_type,,,,,,,,
embedding,286,17387,991257,228.361131,0.798465,0.019825,76.138176,4340.743075


total estimated cost this run: $0.0198


## Next: Stage 4 — Milvus + three indexes (FLAT / HNSW / IVF_FLAT)

Reads `data/chunks/chunks.jsonl`. Each chunk's `embedding` (dimension = `EMBEDDING_DIM`, printed above) gets inserted into Milvus, once per index type, so latency and retrieval accuracy can be benchmarked independently in Stage 5.
